In [0]:
companies_df = spark.read.csv(
    "/Volumes/workspace/default/my_volume/companies_noisy_734.csv",
    header=True,
    inferSchema=True
)

In [0]:
# Company_Size
from pyspark.sql import functions as F

# T1 — Normalisation de la casse
companies_df = companies_df.withColumn(
    "Company_Size",
    F.initcap(F.trim(F.lower(F.col("Company_Size"))))
)

# Validation
companies_df.groupBy("Company_Size").count().orderBy(F.desc("count")).show()

+------------+-----+
|Company_Size|count|
+------------+-----+
|       Small|  528|
|      Medium|  161|
|       Large|   45|
+------------+-----+



In [0]:
# Contract_Status
from pyspark.sql import functions as F
companies_df = companies_df.withColumn(
    "Contract_Status",
    F.initcap(F.trim(F.lower(F.col("Contract_Status"))))
)

# Validation
companies_df.groupBy("Contract_Status").count().orderBy(F.desc("count")).show()


+---------------+-----+
|Contract_Status|count|
+---------------+-----+
|         Active|  528|
|        Pending|  135|
|        Expired|   71|
+---------------+-----+



In [0]:
# Industry
from pyspark.sql import functions as F
from pyspark.sql.window import Window

industry_map = {
    "oil & gas": "Oil & gas",
    "utilities": "Utilities", "utiliies": "Utilities",
    "utilitis": "Utilities", "utilitces": "Utilities", "utilties": "Utilities",
    "machine building": "Machine building",
    "mawhine building": "Machine building", "machie building": "Machine building",
    "machine buildinl": "Machine building", "machine buildng": "Machine building",
    "mchine building": "Machine building",
    "residential": "Residential", "residentdal": "Residential",
    "space": "Space", "spabe": "Space",
    "vehicles": "Vehicles", "vehcles": "Vehicles", "vehicley": "Vehicles",
    "aerospace": "Aerospace", "aerospac": "Aerospace",
    "buildings": "Buildings", "buldings": "Buildings",
    "food & beverage": "Food & beverage", "fod & beverage": "Food & beverage",
    "data centres": "Data centres",
    "healthcare": "Healthcare",
    "mining, metals & minerals": "Mining, metals & minerals",
}

expr = F.col("Industry")
for typo, correct in industry_map.items():
    expr = F.when(
        F.lower(F.trim(F.col("Industry"))) == typo, correct
    ).otherwise(expr)

companies_df = companies_df.withColumn("Industry", expr)

# Validation
companies_df.groupBy("Industry").count().orderBy(F.desc("count")).show(20, truncate=False)

+-------------------------+-----+
|Industry                 |count|
+-------------------------+-----+
|Aerospace                |72   |
|Space                    |65   |
|Mining, metals & minerals|65   |
|Healthcare               |65   |
|Oil & gas                |63   |
|Vehicles                 |62   |
|Machine building         |61   |
|Food & beverage          |61   |
|Residential              |60   |
|Utilities                |60   |
|Buildings                |53   |
|Data centres             |47   |
+-------------------------+-----+



In [0]:
# Payment_Behavior
from pyspark.sql import functions as F
from pyspark.sql.window import Window
# T1 — Normalisation de la casse avec mapping explicite
companies_df = companies_df.withColumn(
    "Payment_Behavior",
    F.when(
        F.lower(F.trim(F.col("Payment_Behavior"))).isin(["on-time", "on_time"]), "On-time"
    ).when(
        F.lower(F.trim(F.col("Payment_Behavior"))).isin(["occasional delay", "occasional_delay"]), "Occasional delay"
    ).when(
        F.lower(F.trim(F.col("Payment_Behavior"))) == "late", "Late"
    ).otherwise(F.col("Payment_Behavior"))
)

# T4 — Imputation des nulls par mode par groupe
count_window = Window.partitionBy("Industry", "Company_Size", "Payment_Behavior")
companies_df = companies_df.withColumn(
    "val_count",
    F.count("Payment_Behavior").over(count_window)
)

group_window = Window.partitionBy("Industry", "Company_Size") \
                     .orderBy(F.desc("val_count"))

companies_df = companies_df.withColumn(
    "mode_val",
    F.first("Payment_Behavior", ignorenulls=True).over(group_window)
)

companies_df = companies_df.withColumn(
    "Payment_Behavior",
    F.when(
        F.col("Payment_Behavior").isNull(), F.col("mode_val")
    ).otherwise(F.col("Payment_Behavior"))
)

# Fallback mode global
companies_df = companies_df.withColumn(
    "Payment_Behavior",
    F.when(
        F.col("Payment_Behavior").isNull(), "On-time"
    ).otherwise(F.col("Payment_Behavior"))
)

# Nettoyage
companies_df = companies_df.drop("val_count", "mode_val")

# Validation
companies_df.groupBy("Payment_Behavior").count().orderBy(F.desc("count")).show()

+----------------+-----+
|Payment_Behavior|count|
+----------------+-----+
|         On-time|  535|
|Occasional delay|  136|
|            Late|   63|
+----------------+-----+



In [0]:
# Campaign_Type
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Mapping direct typo → valeur correcte (en lowercase)
campaign_map = {
    "sem": "SEM", "gem": "SEM", "zem": "SEM", "se": "SEM", "em": "SEM",
    "email": "Email", "emal": "Email",
    "linkedin ads": "LinkedIn Ads", "inkedin ads": "LinkedIn Ads", "lnkedin ads": "LinkedIn Ads",
    "webinar": "Webinar", "ebinar": "Webinar", "webinak": "Webinar",
    "trade show": "Trade Show", "trade shpw": "Trade Show",
    "content marketing": "Content Marketing", "contenv marketing": "Content Marketing",
}

# Créer une map Spark directement
mapping_expr = F.create_map([F.lit(x) for pair in campaign_map.items() for x in pair])

companies_df = companies_df.withColumn(
    "Campaign_Type",
    F.coalesce(
        mapping_expr[F.lower(F.trim(F.col("Campaign_Type")))],
        F.col("Campaign_Type")
    )
)

# Validation
companies_df.groupBy("Campaign_Type").count().orderBy(F.desc("count")).show(truncate=False)

+-----------------+-----+
|Campaign_Type    |count|
+-----------------+-----+
|Content Marketing|139  |
|Email            |133  |
|Webinar          |128  |
|LinkedIn Ads     |116  |
|Trade Show       |114  |
|SEM              |104  |
+-----------------+-----+



In [0]:
# Marketing_Spend (K₺)
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType

# T3 — Nettoyage Marketing_Spend (K₺)
companies_df = companies_df.withColumn(
    "Marketing_Spend (K₺)",
    F.when(
        F.col("Marketing_Spend (K₺)").rlike("(?i)^[0-9]+K$"),
        F.regexp_replace(F.col("Marketing_Spend (K₺)"), "(?i)K", "").cast(IntegerType())
    ).when(
        F.col("Marketing_Spend (K₺)").cast(IntegerType()) > 999,
        (F.col("Marketing_Spend (K₺)").cast(IntegerType()) / 1000).cast(IntegerType())
    ).otherwise(
        F.col("Marketing_Spend (K₺)").cast(IntegerType())
    )
)

# T4 — Imputation des nulls par médiane par groupe
window = Window.partitionBy("Industry", "Company_Size")

companies_df = companies_df.withColumn(
    "Marketing_Spend (K₺)",
    F.when(
        F.col("Marketing_Spend (K₺)").isNull(),
        F.percentile_approx("Marketing_Spend (K₺)", 0.5).over(window)
    ).otherwise(F.col("Marketing_Spend (K₺)"))
)

# T5 — Cast final en int
companies_df = companies_df.withColumn(
    "Marketing_Spend (K₺)",
    F.col("Marketing_Spend (K₺)").cast(IntegerType())
)

# Validation
print("Nulls résiduels :", companies_df.filter(F.col("Marketing_Spend (K₺)").isNull()).count())
companies_df.select("Marketing_Spend (K₺)").distinct().orderBy("Marketing_Spend (K₺)").show(50)

Nulls résiduels : 0
+--------------------+
|Marketing_Spend (K₺)|
+--------------------+
|                   5|
|                   6|
|                   7|
|                   8|
|                   9|
|                  10|
|                  11|
|                  12|
|                  13|
|                  14|
|                  15|
|                  16|
|                  17|
|                  18|
|                  19|
|                  20|
|                  21|
|                  22|
|                  23|
|                  24|
|                  25|
|                  26|
|                  27|
|                  28|
|                  29|
|                  30|
|                  31|
|                  32|
|                  33|
|                  34|
|                  35|
|                  36|
|                  37|
|                  38|
|                  39|
|                  40|
|                  41|
|                  42|
|                  43|
|             

In [0]:
# Annual Revenue
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# T4 — Imputation par médiane par groupe
window = Window.partitionBy("Industry", "Company_Size")

companies_df = companies_df.withColumn(
    "Annual_Revenue (M₺)",
    F.when(
        F.col("Annual_Revenue (M₺)").isNull(),
        F.percentile_approx("Annual_Revenue (M₺)", 0.5).over(window)
    ).otherwise(F.col("Annual_Revenue (M₺)"))
)

# Validation
print("Nulls résiduels :", companies_df.filter(F.col("Annual_Revenue (M₺)").isNull()).count())
companies_df.select("Annual_Revenue (M₺)").describe().show()

Nulls résiduels : 0
+-------+-------------------+
|summary|Annual_Revenue (M₺)|
+-------+-------------------+
|  count|                734|
|   mean| 18.326975476839234|
| stddev|   32.2867799699217|
|    min|                1.0|
|    max|              199.4|
+-------+-------------------+



In [0]:
# Leads_Generated
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType

# T4 — Imputation par médiane par groupe
window = Window.partitionBy("Industry", "Company_Size")

companies_df = companies_df.withColumn(
    "Leads_Generated",
    F.when(
        F.col("Leads_Generated").isNull(),
        F.percentile_approx("Leads_Generated", 0.5).over(window)
    ).otherwise(F.col("Leads_Generated"))
)

# T5 — Cast en int
companies_df = companies_df.withColumn(
    "Leads_Generated",
    F.col("Leads_Generated").cast(IntegerType())
)

# Validation
print("Nulls résiduels :", companies_df.filter(F.col("Leads_Generated").isNull()).count())
companies_df.select("Leads_Generated").describe().show()

Nulls résiduels : 0
+-------+-----------------+
|summary|  Leads_Generated|
+-------+-----------------+
|  count|              734|
|   mean|6.963215258855586|
| stddev|3.167451885794876|
|    min|                2|
|    max|               13|
+-------+-----------------+



In [0]:
# Conversion_Rate (%)
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# T4 — Imputation par médiane par groupe
window = Window.partitionBy("Industry", "Company_Size")

companies_df = companies_df.withColumn(
    "Conversion_Rate (%)",
    F.when(
        F.col("Conversion_Rate (%)").isNull(),
        F.percentile_approx("Conversion_Rate (%)", 0.5).over(window)
    ).otherwise(F.col("Conversion_Rate (%)"))
)

# Validation
print("Nulls résiduels :", companies_df.filter(F.col("Conversion_Rate (%)").isNull()).count())
companies_df.select("Conversion_Rate (%)").describe().show()

Nulls résiduels : 0
+-------+-------------------+
|summary|Conversion_Rate (%)|
+-------+-------------------+
|  count|                734|
|   mean| 0.8023160762942777|
| stddev|0.33427664705617877|
|    min|                0.0|
|    max|                1.5|
+-------+-------------------+



In [0]:
# Days_Since_Last_Purchase

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType

# Étape 1 : remplacer 'N/A' par null
companies_df = companies_df.withColumn(
    "Days_Since_Last_Purchase",
    F.when(
        F.trim(F.col("Days_Since_Last_Purchase")) == "N/A", None
    ).otherwise(F.col("Days_Since_Last_Purchase"))
)

# Étape 2 : cast en double
companies_df = companies_df.withColumn(
    "Days_Since_Last_Purchase",
    F.col("Days_Since_Last_Purchase").cast("double")
)

# Étape 3 : imputation par médiane par groupe
window = Window.partitionBy("Industry", "Company_Size")

companies_df = companies_df.withColumn(
    "Days_Since_Last_Purchase",
    F.when(
        F.col("Days_Since_Last_Purchase").isNull(),
        F.round(F.percentile_approx("Days_Since_Last_Purchase", 0.5).over(window))
    ).otherwise(F.col("Days_Since_Last_Purchase"))
)

# Étape 4 : cast en int
companies_df = companies_df.withColumn(
    "Days_Since_Last_Purchase",
    F.col("Days_Since_Last_Purchase").cast(IntegerType())
)

# Validation
print("Nulls résiduels :", companies_df.filter(F.col("Days_Since_Last_Purchase").isNull()).count())
companies_df.select("Days_Since_Last_Purchase").describe().show()

Nulls résiduels : 0
+-------+------------------------+
|summary|Days_Since_Last_Purchase|
+-------+------------------------+
|  count|                     734|
|   mean|      101.78201634877384|
| stddev|      109.20828268799659|
|    min|                       5|
|    max|                     448|
+-------+------------------------+



In [0]:
# Total_Purchases_Last_Year
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType

# T4 — Imputation par médiane par groupe
window = Window.partitionBy("Industry", "Company_Size")

companies_df = companies_df.withColumn(
    "Total_Purchases_Last_Year",
    F.col("Total_Purchases_Last_Year").cast("double")
)

companies_df = companies_df.withColumn(
    "Total_Purchases_Last_Year",
    F.when(
        F.col("Total_Purchases_Last_Year").isNull(),
        F.round(F.percentile_approx("Total_Purchases_Last_Year", 0.5).over(window))
    ).otherwise(F.col("Total_Purchases_Last_Year"))
)

# T5 — Cast en int (comptage entier)
companies_df = companies_df.withColumn(
    "Total_Purchases_Last_Year",
    F.col("Total_Purchases_Last_Year").cast(IntegerType())
)

# Validation
print("Nulls résiduels :", companies_df.filter(F.col("Total_Purchases_Last_Year").isNull()).count())
companies_df.select("Total_Purchases_Last_Year").distinct().orderBy("Total_Purchases_Last_Year").show()

Nulls résiduels : 0
+-------------------------+
|Total_Purchases_Last_Year|
+-------------------------+
|                        0|
|                        1|
|                        2|
|                        3|
|                        4|
|                        5|
|                        6|
|                        8|
|                        9|
|                       10|
|                       11|
|                       12|
|                       13|
|                       14|
|                       15|
+-------------------------+



In [0]:
# Region
from pyspark.sql import functions as F
from pyspark.sql.window import Window

region_map = {
    # Adana
    "adana": "Adana", "dana": "Adana",
    # Sakarya
    "sakarya": "Sakarya",
    # Ankara
    "ankara": "Ankara", "ankaa": "Ankara",
    # Eskisehir
    "eskisehir": "Eskisehir",
    # Zonguldak
    "zonguldak": "Zonguldak", "onguldak": "Zonguldak",
    "zonguqdak": "Zonguldak",
    # Kayseri
    "kayseri": "Kayseri", "kayeri": "Kayseri",
    # Kocaeli
    "kocaeli": "Kocaeli",
    # Izmir
    "izmir": "Izmir", "ezmir": "Izmir", "izmr": "Izmir",
    # Konya
    "konya": "Konya", "kona": "Konya",
    # Istanbul
    "istanbul": "Istanbul", "istqnbul": "Istanbul",
    # Antalya
    "antalya": "Antalya", "antala": "Antalya",
    # Bursa
    "bursa": "Bursa", "busa": "Bursa",
    "eursa": "Bursa", "jursa": "Bursa",
}

expr = F.col("Region")
for typo, correct in region_map.items():
    expr = F.when(
        F.lower(F.trim(F.col("Region"))) == typo, correct
    ).otherwise(expr)

companies_df = companies_df.withColumn("Region", expr)

# Validation
companies_df.groupBy("Region").count().orderBy(F.desc("count")).show(truncate=False)

+---------+-----+
|Region   |count|
+---------+-----+
|Bursa    |72   |
|Konya    |70   |
|Istanbul |67   |
|Kayseri  |66   |
|Eskisehir|64   |
|Kocaeli  |60   |
|Adana    |60   |
|Antalya  |60   |
|Ankara   |58   |
|Izmir    |56   |
|Zonguldak|55   |
|Sakarya  |46   |
+---------+-----+



In [0]:
# District 
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# T1 + T2 — Normalisation et correction des typos District
district_map = {
    # Izmit
    "izmit": "Izmit", "ihmit": "Izmit", "izmt": "Izmit",
    # Muratpasa
    "muratpasa": "Muratpasa", "muratpasc": "Muratpasa",
    # Melikgazi
    "melikgazi": "Melikgazi", "melikazi": "Melikgazi", "mlikgazi": "Melikgazi",
    # Serdivan
    "serdivan": "Serdivan", "serdkvan": "Serdivan", "setdivan": "Serdivan",
    # Osmangazi
    "osmangazi": "Osmangazi", "osmaogazi": "Osmangazi", "osmanazi": "Osmangazi",
    # Seyhan
    "seyhan": "Seyhan", "seyhin": "Seyhan",
    # Kepez
    "kepez": "Kepez", "kepbz": "Kepez",
    # Beylikduzu
    "beylikduzu": "Beylikduzu",
    # Kozlu
    "kozlu": "Kozlu", "koblu": "Kozlu",
    # Meram
    "meram": "Meram", "meam": "Meram",
    # Gebze
    "gebze": "Gebze", "ebze": "Gebze",
    # Tepebasi
    "tepebasi": "Tepebasi", "tpebasi": "Tepebasi",
    # Yenimahalle
    "yenimahalle": "Yenimahalle", "yenimahclle": "Yenimahalle",
    # Konyaalti
    "konyaalti": "Konyaalti",
    # autres valeurs correctes
    "ceyhan": "Ceyhan", "arifiye": "Arifiye",
    "etimesgut": "Etimesgut", "izmit": "Izmit",
    "cigli": "Cigli", "korfez": "Korfez",
    "karsiyaka": "Karsiyaka", "atasehir": "Atasehir",
    "konak": "Konak", "cankaya": "Cankaya",
    "selcuklu": "Selcuklu", "sariyer": "Sariyer",
    "kartal": "Kartal", "odunpazari": "Odunpazari",
    "kocasinan": "Kocasinan", "nilufer": "Nilufer",
    "mamak": "Mamak", "yildirim": "Yildirim",
    "eregli": "Eregli", "umraniye": "Umraniye",
    "bornova": "Bornova",
}

expr = F.col("District")
for typo, correct in district_map.items():
    expr = F.when(
        F.lower(F.trim(F.col("District"))) == typo, correct
    ).otherwise(expr)

companies_df = companies_df.withColumn("District", expr)

# T4 — Imputation des 3 nulls par mode par groupe Region
count_window = Window.partitionBy("Region", "District")
companies_df = companies_df.withColumn(
    "val_count",
    F.count("District").over(count_window)
)

group_window = Window.partitionBy("Region").orderBy(F.desc("val_count"))
companies_df = companies_df.withColumn(
    "mode_val",
    F.first("District", ignorenulls=True).over(group_window)
)

companies_df = companies_df.withColumn(
    "District",
    F.when(
        F.col("District").isNull(), F.col("mode_val")
    ).otherwise(F.col("District"))
)

# Nettoyage
companies_df = companies_df.drop("val_count", "mode_val")

# Validation
print("Nulls résiduels :", companies_df.filter(F.col("District").isNull()).count())
companies_df.groupBy("District").count().orderBy(F.desc("count")).show(40, truncate=False)

Nulls résiduels : 0
+-----------+-----+
|District   |count|
+-----------+-----+
|Kocasinan  |40   |
|Seyhan     |38   |
|Selcuklu   |38   |
|Tepebasi   |32   |
|Odunpazari |32   |
|Meram      |32   |
|Nilufer    |30   |
|Arifiye    |30   |
|Kozlu      |29   |
|Kepez      |28   |
|Eregli     |26   |
|Melikgazi  |26   |
|Osmangazi  |23   |
|Gebze      |23   |
|Ceyhan     |22   |
|Izmit      |20   |
|Konak      |20   |
|Yildirim   |19   |
|Atasehir   |18   |
|Yenimahalle|18   |
|Korfez     |17   |
|Cankaya    |16   |
|Konyaalti  |16   |
|Muratpasa  |16   |
|Serdivan   |16   |
|Cigli      |15   |
|Sariyer    |15   |
|Kartal     |13   |
|Etimesgut  |12   |
|Mamak      |12   |
|Umraniye   |11   |
|Karsiyaka  |11   |
|Beylikduzu |10   |
|Bornova    |10   |
+-----------+-----+



In [0]:
# Preferred_Channel 
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# T1 — Normalisation avec mapping explicite
companies_df = companies_df.withColumn(
    "Preferred_Channel",
    F.when(
        F.lower(F.trim(F.col("Preferred_Channel"))) == "sales rep", "Sales Rep"
    ).when(
        F.lower(F.trim(F.col("Preferred_Channel"))) == "online", "Online"
    ).when(
        F.lower(F.trim(F.col("Preferred_Channel"))) == "email", "Email"
    ).when(
        F.lower(F.trim(F.col("Preferred_Channel"))) == "dealer", "Dealer"
    ).otherwise(F.col("Preferred_Channel"))
)

# T4 — Imputation des 4 nulls par mode par groupe
count_window = Window.partitionBy("Industry", "Company_Size", "Preferred_Channel")
companies_df = companies_df.withColumn(
    "val_count",
    F.count("Preferred_Channel").over(count_window)
)

group_window = Window.partitionBy("Industry", "Company_Size") \
                     .orderBy(F.desc("val_count"))

companies_df = companies_df.withColumn(
    "mode_val",
    F.first("Preferred_Channel", ignorenulls=True).over(group_window)
)

companies_df = companies_df.withColumn(
    "Preferred_Channel",
    F.when(
        F.col("Preferred_Channel").isNull(), F.col("mode_val")
    ).otherwise(F.col("Preferred_Channel"))
)

# Fallback mode global
companies_df = companies_df.withColumn(
    "Preferred_Channel",
    F.when(
        F.col("Preferred_Channel").isNull(), "Sales Rep"
    ).otherwise(F.col("Preferred_Channel"))
)

# Nettoyage
companies_df = companies_df.drop("val_count", "mode_val")

# Validation
companies_df.groupBy("Preferred_Channel").count().orderBy(F.desc("count")).show()

+-----------------+-----+
|Preferred_Channel|count|
+-----------------+-----+
|        Sales Rep|  318|
|           Dealer|  201|
|           Online|  150|
|            Email|   65|
+-----------------+-----+



In [0]:
# 1. Vérifier les nulls sur toutes les colonnes
from pyspark.sql.functions import col, count, when

null_counts = companies_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in companies_df.columns
])

null_counts.show(vertical=True)

-RECORD 0------------------------
 Company_ID                | 0   
 Industry                  | 0   
 Company_Size              | 0   
 Annual_Revenue (M₺)       | 0   
 Marketing_Spend (K₺)      | 0   
 Campaign_Type             | 0   
 Leads_Generated           | 0   
 Conversion_Rate (%)       | 0   
 Region                    | 0   
 District                  | 0   
 Last_Product_1            | 0   
 Last_Product_2            | 0   
 Frequency_of_Purchase     | 0   
 Days_Since_Last_Purchase  | 0   
 Contract_Status           | 0   
 Total_Purchases_Last_Year | 0   
 Payment_Behavior          | 0   
 Preferred_Channel         | 0   
 Sales_Rep                 | 0   



In [0]:
# 2. Vérifier le schéma
companies_df.printSchema()

root
 |-- Company_ID: string (nullable = true)
 |-- Industry: string (nullable = true)
 |-- Company_Size: string (nullable = true)
 |-- Annual_Revenue (M₺): double (nullable = true)
 |-- Marketing_Spend (K₺): integer (nullable = true)
 |-- Campaign_Type: string (nullable = true)
 |-- Leads_Generated: integer (nullable = true)
 |-- Conversion_Rate (%): double (nullable = true)
 |-- Region: string (nullable = true)
 |-- District: string (nullable = true)
 |-- Last_Product_1: string (nullable = true)
 |-- Last_Product_2: string (nullable = true)
 |-- Frequency_of_Purchase: string (nullable = true)
 |-- Days_Since_Last_Purchase: integer (nullable = true)
 |-- Contract_Status: string (nullable = true)
 |-- Total_Purchases_Last_Year: integer (nullable = true)
 |-- Payment_Behavior: string (nullable = true)
 |-- Preferred_Channel: string (nullable = true)
 |-- Sales_Rep: string (nullable = true)



In [0]:
# 3. Vérifier les stats numériques
companies_df.select(
    "Annual_Revenue (M₺)",
    "Marketing_Spend (K₺)",
    "Leads_Generated",
    "Conversion_Rate (%)",
    "Days_Since_Last_Purchase",
    "Total_Purchases_Last_Year"
).describe().show()

+-------+-------------------+--------------------+-----------------+-------------------+------------------------+-------------------------+
|summary|Annual_Revenue (M₺)|Marketing_Spend (K₺)|  Leads_Generated|Conversion_Rate (%)|Days_Since_Last_Purchase|Total_Purchases_Last_Year|
+-------+-------------------+--------------------+-----------------+-------------------+------------------------+-------------------------+
|  count|                734|                 734|              734|                734|                     734|                      734|
|   mean| 18.326975476839234|   21.29019073569482|6.963215258855586| 0.8023160762942777|      101.78201634877384|        6.640326975476839|
| stddev|   32.2867799699217|  10.067384790780766|3.167451885794876|0.33427664705617877|      109.20828268799659|        4.657292880357893|
|    min|                1.0|                   5|                2|                0.0|                       5|                        0|
|    max|           

In [0]:
# 4. Vérifier les colonnes catégorielles
for c in ["Industry", "Company_Size", "Contract_Status",
          "Payment_Behavior", "Campaign_Type", "Region",
          "Preferred_Channel"]:
    print(f"\n=== {c} ===")
    companies_df.groupBy(c).count().orderBy(F.desc("count")).show(truncate=False)


=== Industry ===
+-------------------------+-----+
|Industry                 |count|
+-------------------------+-----+
|Aerospace                |72   |
|Space                    |65   |
|Mining, metals & minerals|65   |
|Healthcare               |65   |
|Oil & gas                |63   |
|Vehicles                 |62   |
|Machine building         |61   |
|Food & beverage          |61   |
|Residential              |60   |
|Utilities                |60   |
|Buildings                |53   |
|Data centres             |47   |
+-------------------------+-----+


=== Company_Size ===
+------------+-----+
|Company_Size|count|
+------------+-----+
|Small       |528  |
|Medium      |161  |
|Large       |45   |
+------------+-----+


=== Contract_Status ===
+---------------+-----+
|Contract_Status|count|
+---------------+-----+
|Active         |528  |
|Pending        |135  |
|Expired        |71   |
+---------------+-----+


=== Payment_Behavior ===
+----------------+-----+
|Payment_Behavior|coun

In [0]:
# Nulls
null_counts.show(vertical=True)

-RECORD 0------------------------
 Company_ID                | 0   
 Industry                  | 0   
 Company_Size              | 0   
 Annual_Revenue (M₺)       | 0   
 Marketing_Spend (K₺)      | 0   
 Campaign_Type             | 0   
 Leads_Generated           | 0   
 Conversion_Rate (%)       | 0   
 Region                    | 0   
 District                  | 0   
 Last_Product_1            | 0   
 Last_Product_2            | 0   
 Frequency_of_Purchase     | 0   
 Days_Since_Last_Purchase  | 0   
 Contract_Status           | 0   
 Total_Purchases_Last_Year | 0   
 Payment_Behavior          | 0   
 Preferred_Channel         | 0   
 Sales_Rep                 | 0   



In [0]:
import os

# Étape 1 : sauvegarder
companies_df.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("/Volumes/workspace/default/my_volume/companies_clean_temp")

# Étape 2 : trouver le fichier part-00000
files = dbutils.fs.ls("/Volumes/workspace/default/my_volume/companies_clean_temp")
part_file = [f.path for f in files if f.name.startswith("part-")][0]

# Étape 3 : renommer vers le nom souhaité
dbutils.fs.cp(
    part_file,
    "/Volumes/workspace/default/my_volume/companies_clean_final.csv"
)

# Étape 4 : supprimer le dossier temporaire
dbutils.fs.rm("/Volumes/workspace/default/my_volume/companies_clean_temp", recurse=True)

# Validation
display(dbutils.fs.ls("/Volumes/workspace/default/my_volume/"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/my_volume/companies_clean_final.csv,companies_clean_final.csv,121559,1778112240000
dbfs:/Volumes/workspace/default/my_volume/companies_noisy_734.csv,companies_noisy_734.csv,122165,1776509821000
dbfs:/Volumes/workspace/default/my_volume/employees_clean_final/,employees_clean_final/,0,1778112240749
dbfs:/Volumes/workspace/default/my_volume/employees_noisy_5234.csv,employees_noisy_5234.csv,818844,1776509821000
